# SymPL + ReCLIP | sanity

Kaggle settings: **GPU T4 ×2**, **Internet On**. Add Dataset **ltrnnguyn/sc203-apc-trace-for-ablation**. The notebook finds the existing trace bundle in Kaggle Input. Set trace_input only if needed. HF_token is optional for Hugging Face downloads.

**Three questions**, one per task. Run this before the full notebook. Sanity passes when both selectors produce predictions for all three questions and at least one object has multiple ranked proposals.

Runs both selectors on the same frozen inputs and proposal order. Original box selection is replayed in the same environment. The historical answer is saved separately to measure baseline drift. Only ReCLIP isolated proposal scoring is enabled: crop and blur, one OpenAI CLIP ViT-B/32 backbone, no relation resolver.

Historical SymPL calls and depth maps are fetched from the public saved Kaggle output. You can optionally set sympl_trace to a local copy of that output if the public API is unavailable.

Open gallery.html in /kaggle/working/sympl_reclip_sanity. Download the full output directory with its images to retain their links. The diagnostic cohort is not the full benchmark. Grounding accuracy requires confirmed box annotations.


In [ ]:
REPO_URL = 'https://github.com/leetranwyng/sc203-kat.git'
CODE_REF = 'v0.4.0-reclip'
HF_SECRET_NAME = "HF_token"
RUN_CONFIG = {'method': 'sympl',
 'mode': 'sanity',
 'selector': 'paired',
 'sample_ids': ['5832', '4798', '8268'],
 'seed': 42,
 'trace_input': None,
 'model_input': None,
 'grounding_labels': None,
 'runtime_dir': '/kaggle/temp/sc203_reclip/sympl_reclip_sanity',
 'output_dir': '/kaggle/working/sympl_reclip_sanity',
 'sympl_trace': None}


In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

if not CODE_REF or CODE_REF.startswith('-') or any(c.isspace() for c in CODE_REF):
    raise ValueError('CODE_REF must be a Git tag, branch, or full commit.')
if REPO_URL != 'https://github.com/leetranwyng/sc203-kat.git':
    raise ValueError('Use the public leetranwyng/sc203-kat repository.')
GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT='0', GCM_INTERACTIVE='Never')
cache = Path(os.environ.get('SC203_CODE_CACHE', '/kaggle/temp/sc203-kat-code'))
cache.mkdir(parents=True, exist_ok=True)
key = hashlib.sha256((REPO_URL + '\n' + CODE_REF).encode()).hexdigest()[:16]
CODE_ROOT = cache / key

def git(*args):
    result = subprocess.run(['git', *args], env=GIT_ENV, text=True,
                            capture_output=True, timeout=300)
    if result.returncode: raise RuntimeError((result.stderr or result.stdout).strip())
    return result.stdout.strip()

if not (CODE_ROOT / '.git').exists():
    if CODE_ROOT.exists() and any(CODE_ROOT.iterdir()):
        raise RuntimeError('Code cache is not empty; choose a different SC203_CODE_CACHE.')
    git('clone', '--no-checkout', '--no-tags', '--depth', '1', REPO_URL, str(CODE_ROOT))
else:
    if git('-C', str(CODE_ROOT), 'remote', 'get-url', 'origin') != REPO_URL:
        raise RuntimeError('Cached source origin differs from REPO_URL.')
    if git('-C', str(CODE_ROOT), 'status', '--porcelain', '--untracked-files=normal'):
        raise RuntimeError('Cached source has local changes; use a new code cache.')
git('-C', str(CODE_ROOT), 'fetch', '--depth', '1', '--no-tags', 'origin', CODE_REF)
commit = git('-C', str(CODE_ROOT), 'rev-parse', '--verify', 'FETCH_HEAD^{commit}')
git('-C', str(CODE_ROOT), 'checkout', '--detach', commit)

ENV = dict(GIT_ENV, SC203_KAT_ROOT=str(CODE_ROOT), PYTHONPATH=str(CODE_ROOT / 'src'))
if HF_SECRET_NAME and not ENV.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(HF_SECRET_NAME)
        if value: ENV['HF_TOKEN'] = value.strip()
        del value
    except Exception:
        print('HF secret unavailable; using public/local model files.')
config_dir = Path(os.environ.get('SC203_CONFIG_DIR', '/kaggle/temp/sc203-kat-configs'))
config_dir.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = config_dir / (RUN_CONFIG['method'] + '_' + RUN_CONFIG['mode'] + '.json')
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')
prepare = (
    'import json,sys; from pathlib import Path; '
    'from sc203_kat.runtime.kaggle import ensure_submodule,source_metadata; '
    'root=Path(sys.argv[1]); method=sys.argv[2]; '
    'ensure_submodule(root,method); metadata=source_metadata(root,method); '
    'print(json.dumps(metadata,indent=2))'
)
subprocess.run([sys.executable, '-c', prepare, str(CODE_ROOT), RUN_CONFIG['method']], cwd=CODE_ROOT, env=ENV, check=True)
print('Code:', commit, '| Method:', RUN_CONFIG['method'], '| Mode:', RUN_CONFIG['mode'])


In [ ]:
output = Path(RUN_CONFIG['output_dir'])
output.mkdir(parents=True, exist_ok=True)
if RUN_CONFIG['method'] == 'sympl' and not RUN_CONFIG.get('sympl_trace'):
    history = Path(RUN_CONFIG['runtime_dir']) / 'sympl_history'
    command = [sys.executable, '-u', '-m', 'sc203_kat.reclip.fetch_sympl', str(history)]
    if RUN_CONFIG['sample_ids']:
        command += ['--sample-ids', *RUN_CONFIG['sample_ids']]
    subprocess.run(command, cwd=CODE_ROOT, env=ENV, check=True)
    RUN_CONFIG['sympl_trace'] = str(history)
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')


In [ ]:
module = 'sc203_kat.reclip.apc' if RUN_CONFIG['method'] == 'apc_vis' else 'sc203_kat.reclip.sympl'
arguments = ['--config', str(CONFIG_PATH)] if RUN_CONFIG['method'] == 'apc_vis' else [str(CONFIG_PATH)]
worker = subprocess.run([sys.executable, '-u', '-m', module, *arguments], cwd=CODE_ROOT, env=ENV)
report = None
if (output / 'replay_predictions.jsonl').is_file():
    report = subprocess.run([sys.executable, '-u', '-m', 'sc203_kat.reclip.report', '--config', str(CONFIG_PATH)],
                            cwd=CODE_ROOT, env=ENV)
worker.check_returncode()
if report is None:
    raise RuntimeError(f'Runner finished without replay predictions. Inspect {output}.')
report.check_returncode()


In [ ]:
from IPython.display import FileLink, display
for name in ('gallery.html', 'summary.json', 'paired_predictions.csv',
             'grounding_decisions.jsonl', 'grounding_disagreements.jsonl'):
    path = output / name
    if path.is_file():
        display(FileLink(str(path)))
